# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anujrkt06-tech/Flyrank-ML-project-/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

I use February information as the feature window. I use GSC performance, position, AI sessions, and GA4 engagement information that was available before the prediction window. I create CTR and average-position features. Numeric missing values are filled with 0, and GA4 availability is kept as a separate categorical feature.

In [7]:
features = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(COALESCE(gsc_impressions, 0)) AS imp_feb,
    SUM(COALESCE(gsc_clicks, 0)) AS clk_feb,

    SUM(COALESCE(gsc_sum_position, 0)) AS pos_sum_feb,

    SUM(
        CASE WHEN gsc_data_available = TRUE
             THEN COALESCE(sessions_ai, 0)
             ELSE 0 END
    ) AS sessions_ai_feb,

    SUM(
        CASE WHEN ga4_data_available = TRUE
             THEN COALESCE(ga4_pageviews, 0)
             ELSE 0 END
    ) AS ga4_pageviews_feb,

    SUM(
        CASE WHEN ga4_data_available = TRUE
             THEN COALESCE(ga4_sessions, 0)
             ELSE 0 END
    ) AS ga4_sessions_feb,

    SUM(
        CASE WHEN ga4_data_available = TRUE
             THEN COALESCE(ga4_total_engagement_sec, 0)
             ELSE 0 END
    ) AS ga4_engagement_sec_feb,

    MAX(CASE WHEN ga4_data_available = TRUE THEN 1 ELSE 0 END)
        AS ga4_available_feb

FROM {FEB}
GROUP BY client_hash_id, content_hash_id
""").df()

# Engineered features
features["ctr_feb"] = np.where(
    features["imp_feb"] > 0,
    features["clk_feb"] / features["imp_feb"],
    0
)

features["avg_position_feb"] = np.where(
    features["imp_feb"] > 0,
    features["pos_sum_feb"] / features["imp_feb"],
    0
)

# Remove intermediate feature
features = features.drop(columns=["pos_sum_feb"])

# Build model-ready vector
X = features.drop(
    columns=["client_hash_id", "content_hash_id"]
).copy()

# Fill numeric missing values
X = X.fillna(0)

print("Feature vector shape:", X.shape)
print(X.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature vector shape: (321546, 9)
   imp_feb  clk_feb  sessions_ai_feb  ga4_pageviews_feb  ga4_sessions_feb  \
0    299.0      0.0              0.0                0.0               0.0   
1    733.0      6.0              0.0                6.0               6.0   
2    514.0      0.0              0.0                1.0               1.0   
3   2931.0      3.0              0.0                9.0               6.0   
4    970.0      2.0              0.0                3.0               3.0   

   ga4_engagement_sec_feb  ga4_available_feb   ctr_feb  avg_position_feb  
0                     0.0                  0  0.000000         12.448161  
1                     0.0                  1  0.008186          6.316508  
2                     0.0                  1  0.000000          9.966926  
3                   193.0                  1  0.001024         41.814739  
4                     0.0                  1  0.002062         10.307216  


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.